# Phase 8 — SHAP Explainability

The frozen model is a **linear** Logistic Regression, so SHAP values here are computed with `shap.LinearExplainer` — **exact**, not the sampling-based approximation tree/deep models need. Verified in development: `base_value + sum(shap_values) == model.decision_function()` exactly, for every row checked.

**Background/global-summary data:** the 432 unaugmented train patients (one row per real patient, from `clinical_2`) — not `train_pool`'s 2,029 rows. Using the augmented set here would let patients with more `lifestyle.csv` copies dominate the summary plot, the same reasoning Phase 2's EDA used.

**Individual-example data:** `holdout_validation` (109 patients, never trained on) — real predictions on real unseen patients, with Phase 7's already-known true/predicted labels available to pick meaningful examples.

SHAP explains what the model did, not clinical truth — a reminder that carries through every plot below.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.config import INTERIM_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, PATIENT_ID_COL
from src.preprocessing import split_features_target
from src.data_loader import load_clinical_2_full
from src.feature_mapping import harmonize
from src.explain import (
    build_explainer, compute_shap_values, get_preprocessed_frame,
    clean_display_columns, get_top_contributors, build_structured_explanation, categorize_risk,
)

pd.set_option("display.width", 200)

pipeline = joblib.load(MODELS_DIR / "pcos_risk_pipeline.joblib")

patient_split = pd.read_csv(INTERIM_DIR / "patient_split.csv")
c2 = harmonize(load_clinical_2_full())
train_ids = set(patient_split.loc[patient_split["split"] == "train", PATIENT_ID_COL])
train_patients_df = c2[c2[PATIENT_ID_COL].isin(train_ids)].reset_index(drop=True)
X_train_patients, y_train_patients = split_features_target(train_patients_df)

explainer, X_bg = build_explainer(pipeline, X_train_patients)
print("background:", X_bg.shape)

## Global explanation

In [ ]:
X_bg_clean = clean_display_columns(X_bg)
shap_values_global = compute_shap_values(explainer, X_bg_clean)

fig = plt.figure(figsize=(8, 8))
shap.plots.beeswarm(shap_values_global, show=False, max_display=15)
plt.title("SHAP summary — direction and magnitude of feature influence", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "19_shap_beeswarm.png", dpi=160)
plt.show()

**Interpretation:** Follicle No. (R) and Follicle No. (L) dominate — high values (red) push the prediction toward PCOS, low values (blue) push away from it, with almost no overlap. Cycle(R/I)=2.0 (Regular) reduces predicted risk. The three self-reported symptoms (hirsutism, skin darkening, pimples) show a clean binary split: present (red) increases risk, absent (blue) decreases it, each by a smaller but consistent amount. Weight and Marriage Status show weaker, more mixed effects.

**Limitation, repeated deliberately:** the top two features are the same follicle-count variables flagged since Phase 2 as diagnostic-criterion-adjacent (Rotterdam criteria). This plot shows the model relies on them heavily — expected, not a new discovery — which is exactly why the feature-mapping table (Phase 1) and every phase since has carried that caveat forward.

In [ ]:
mean_abs_shap = pd.Series(np.abs(shap_values_global.values).mean(axis=0), index=X_bg_clean.columns)
top15 = mean_abs_shap.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top15.index, top15.values, color="#2a78d6")
ax.set_xlabel("Mean |SHAP value| (average impact on predicted log-odds)")
ax.set_title("Global feature importance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "20_shap_feature_importance.png", dpi=160)
plt.show()

top15.sort_values(ascending=False)

**Direction of influence (from the beeswarm above), for the top features:**

| Feature | Direction |
|---|---|
| Follicle No. (R), Follicle No. (L) | Higher → increases predicted risk |
| Cycle(R/I) = 2.0 (Regular) | Present → decreases predicted risk |
| hair growth(Y/N), Skin darkening (Y/N), Pimples(Y/N) | Present (Yes) → increases predicted risk |
| Weight gain(Y/N) | Present → increases predicted risk |
| Marriage Status (Yrs) | Higher → mixed/slightly decreases (weak effect) |
| AMH(ng/mL) | Higher → increases predicted risk |

## Dependence plots for the two dominant features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, feature in zip(axes, ["Follicle No. (R)", "AMH(ng/mL)"]):
    idx = list(X_bg_clean.columns).index(feature)
    ax.scatter(X_bg_clean[feature], shap_values_global.values[:, idx], s=18, alpha=0.6, color="#2a78d6", edgecolors="none")
    ax.axhline(0, color="#c3c2b7", linewidth=1)
    ax.set_xlabel(feature)
    ax.set_ylabel("SHAP value (impact on predicted log-odds)")
    ax.set_title(f"Dependence: {feature}", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "21_shap_dependence_plots.png", dpi=160)
plt.show()

**Interpretation:** Follicle No. (R) shows a clean monotonic relationship — SHAP impact rises steadily as follicle count increases, with a visible steepening above roughly 10-12 (the Rotterdam ultrasound threshold flagged in Phase 2). AMH shows the same increasing-risk direction but a noisier, more gradual relationship, consistent with AMH being a correlate of follicle count rather than the ultrasound measurement itself.

## Individual explanations

Four holdout patients, picked using Phase 7's already-known true/predicted labels: a confident correct high-risk call, a confident correct low-risk call, a confident false positive, and a confident false negative.

In [ ]:
holdout = pd.read_csv(INTERIM_DIR / "holdout_validation.csv")
X_holdout, y_holdout = split_features_target(holdout)
X_holdout_t = get_preprocessed_frame(pipeline, X_holdout)
X_holdout_clean = clean_display_columns(X_holdout_t)
shap_values_holdout = compute_shap_values(explainer, X_holdout_clean)

scored = pd.read_csv(TABLES_DIR / "phase7_holdout_scored.csv")

# positional index within `holdout` for each named example patient (identified from Phase 7's scored predictions)
examples = {
    "Correct high-risk": 27,
    "Correct low-risk": 42,
    "False positive": 378,
    "False negative": 445,
}
example_rows = {
    label: holdout.index[holdout[PATIENT_ID_COL] == pid][0]
    for label, pid in examples.items()
}
example_rows

In [ ]:
from src.explain import build_single_explanation

structured_explanations = {}

for label, row_idx in example_rows.items():
    raw_row = X_holdout.iloc[row_idx]
    shap_row = pd.Series(shap_values_holdout.values[row_idx], index=X_holdout_t.columns)
    y_true = y_holdout.iloc[row_idx]
    pid = examples[label]

    structured_explanations[label] = build_structured_explanation(
        pipeline, explainer, raw_row, X_holdout_t.iloc[row_idx], shap_row, X_holdout_t.columns
    )
    structured_explanations[label]["patient_id"] = int(pid)
    structured_explanations[label]["actual_label"] = "PCOS" if y_true == 1 else "No PCOS"

    # Individual figure per example - shap.plots.waterfall draws its own full-figure
    # layout internally and does not respect being placed into a subplot axis, so
    # each patient gets its own figure rather than a forced 2x2 grid.
    single_exp = build_single_explanation(
        shap_values_holdout, row_idx, raw_row, X_holdout_t.columns, X_holdout_clean.columns
    )
    plt.figure(figsize=(7.5, 5))
    shap.plots.waterfall(single_exp, show=False, max_display=8)
    plt.title(f"{label} — Patient {pid} (actual: {'PCOS' if y_true==1 else 'No PCOS'})", fontsize=10)
    plt.tight_layout()
    safe_label = label.lower().replace(" ", "_")
    plt.savefig(FIGURES_DIR / f"22_shap_waterfall_{safe_label}.png", dpi=150, bbox_inches="tight")
    plt.show()

with open(TABLES_DIR / "phase8_structured_explanations.json", "w") as f:
    json.dump(structured_explanations, f, indent=2, default=str)

print(json.dumps(structured_explanations, indent=2, default=str))

### Reading the four examples

- **Correct high-risk (Patient 27, actual PCOS, predicted 99.99%):** driven almost entirely by very high follicle counts on both ovaries — a textbook case where the model, the diagnostic criteria, and the outcome all agree.
- **Correct low-risk (Patient 42, actual No PCOS, predicted ~0.01%):** driven mainly by a low PRG(ng/mL) reading and the absence of pimples/an irregular cycle. This example is also what surfaced a real robustness gap during this analysis: this patient's raw `PRG(ng/mL)` is 85.0, while `train_pool`'s maximum is 25.3 — a standardized z-score of ~49. Before the fix below, that single value's SHAP impact (-9.8) dwarfed every other feature combined, for a variable that isn't even the model's intended top predictor. **Fix applied:** `src/preprocessing.py` now clips standardized numeric values to +/-5 (`ClipStandardized`), added specifically because of this finding. After the fix, PRG's impact here is -1.3 — still the largest single contributor, but no longer overwhelming the rest of the patient's profile. The final model, holdout metrics, and all figures in this notebook already reflect the fix (Phases 6-8 were rerun after adding it).
- **False positive (Patient 378, actual No PCOS, predicted ~80%):** driven by an irregular cycle, reported pimples, and weight gain — despite normal hirsutism/skin-darkening status pulling the other way. A mixed presentation that reads as ambiguous to the model; worth a closer look at whether this is a genuinely borderline patient or a labeling edge case.
- **False negative (Patient 445, actual PCOS, predicted ~1.5%):** very low follicle counts on both ovaries (value = 1) dominate the prediction toward "No PCOS", outweighing an elevated AMH (16.9) that on its own would suggest otherwise. This is exactly the error type this project's whole design has been oriented around minimizing — a concrete example to walk through in the final report, not just a metric in a table.

## Summary

- SHAP values are exact (linear model), computed on a clean, unaugmented 432-patient background.
- Global explanation confirms Phase 2's EDA almost feature-for-feature: follicle counts and cycle regularity dominate, followed by the three hyperandrogenism/insulin-resistance symptoms and AMH.
- Four individual examples are saved as both waterfall plots and structured JSON (`reports/tables/phase8_structured_explanations.json`) in exactly the shape Phase 9's AI agent will consume.
- The false-negative example is flagged as the concrete illustration of this project's central error-cost tradeoff, worth including directly in the final report rather than only as an aggregate recall number.